# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GEMINI_API_KEY') or os.getenv('OPENAI_API_KEY')
using_gemini = bool(os.getenv('GEMINI_API_KEY'))

if not api_key:
    print("No API key found. Please set GEMINI_API_KEY in your .env file.")
elif using_gemini and not api_key.startswith('AIz'):
    print("Gemini API key found, but it does not look right (expected prefix AIz).")
elif (not using_gemini) and (not api_key.startswith('sk-proj-')):
    print("OpenAI API key found, but it does not look right (expected prefix sk-proj-).")
else:
    provider = "Gemini" if using_gemini else "OpenAI"
    print(f"{provider} API key looks good so far")
    
if using_gemini:
    openai = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key=api_key)
    MODEL = "gemini-2.5-flash-lite"
    MODEL_BROCHURE = "gemini-2.5-flash"
else:
    openai = OpenAI(api_key=api_key)
    MODEL = "gpt-5-nano"
    MODEL_BROCHURE = "gpt-4.1-mini"

Gemini API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company profile', 'url': 'https://edwarddonner.com/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'social media', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social media', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'social media',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gemini-2.5-flash-lite
Found 6 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company information', 'url': 'https://edwarddonner.com/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'social media', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social media', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'social media',
   'url': 'https://www.facebook.com/edward.donner.52/'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash-lite
Found 9 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/about'},
  {'type': 'models page', 'url': 'https://huggingface.co/models'},
  {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'documentation page', 'url': 'https://huggingface.co/docs'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash-lite
Found 13 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
NEW
Google Gemma 4 is here 💫
Storage Buckets: AI-native object storage
GGML and llama.cpp join Hugging Face 🔥
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
google/gemma-4-31B-it
Updated
5 days ago
•
884k
•
1.24k
Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled
Updated
1 day ago
•
552k
•
2.43k
dealignai/Gemma-4-31B-JANG_4M-CRACK
Updated
3 days ago
•
29.5k
•
598
netflix/void-model
Updated
about 17 hours ago
•
500
prism-ml/Bonsai-8B-gguf
Updated
1 day ago
•
52.6k
•
486
Browse 2M+ models
Spaces
Running
on
Zero
Featured
222
OmniVoice
🌍
222
High-quality voice clon

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash-lite
Found 10 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nNEW\nGoogle Gemma 4 is here 💫\nStorage Buckets: AI-native object storage\nGGML and llama.cpp join Hugging Face 🔥\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ngoogle/gemma-4-31B-it\nUpdated\n5 days ago\n•\n884k\n•\n1.24k\nJackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled\nUpdated\n1 day ago\n•\n552k\n•\n2.43k\ndealignai/Gemma-4-31B-JANG_4M-CRACK\nUpdated\n3 days ago\n•\n29.5k\n•\n598\nnetflix/void-model\nUpdated\nabout 17 ho

In [17]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL_BROCHURE,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash-lite
Found 10 relevant links


**Hugging Face: The AI Community Building the Future.**

Welcome to Hugging Face, the leading platform empowering the global machine learning community to collaborate, innovate, and accelerate the future of artificial intelligence. We are dedicated to fostering an open and collaborative environment, providing the essential tools and infrastructure for groundbreaking ML development.

---

**What We Offer:**

Hugging Face is the central hub for all things machine learning, offering a comprehensive ecosystem for creators, researchers, and developers:

*   **Models:** Explore and share from over **2 million models**, covering a vast array of tasks including text generation, image manipulation, speech synthesis, and more. Our platform supports diverse libraries like PyTorch, TensorFlow, and Transformers, alongside various inference providers.
*   **Datasets:** Access and contribute to over **500,000 datasets**, providing the foundational data to train and refine the next generation of AI applications.
*   **Spaces (Applications):** Deploy, demonstrate, and interact with over **1 million AI applications**. Discover cutting-edge tools ranging from high-quality voice cloning in 600+ languages to advanced image and video generation, all built by the community.
*   **Buckets:** Leverage our **AI-native object storage**, specifically designed to meet the unique demands of machine learning workloads.
*   **Open Source Excellence:** Accelerate your innovation with our robust open-source stack. We enable unlimited collaboration on public models, datasets, and applications, embracing and integrating pivotal technologies like GGML and llama.cpp.

We support all modalities, from text and image to audio and video, ensuring a versatile platform for every AI endeavor.

---

**Our Community & Culture:**

At its core, Hugging Face thrives on **community and collaboration**. We believe in an open, inclusive, and fast-paced environment where knowledge sharing, collective building, and pushing the boundaries of AI are paramount. We foster a culture where innovation is a shared journey, empowering every individual to contribute meaningfully to the advancement of machine learning. Join us in shaping the AI revolution!

---

**For Prospective Customers:**
Accelerate your AI initiatives with unparalleled access to the world's largest repository of ML assets and a powerful, collaborative platform designed for rapid development. Explore our enterprise solutions tailored to scale your innovation effectively.

**For Investors:**
Invest in the foundational infrastructure of the AI era. Hugging Face stands as the central hub for machine learning development, driving widespread adoption and transformative innovation across all industries.

**For Prospective Recruits:**
Become part of a vibrant and visionary team that is actively building the future of AI. Join a culture that champions open-source principles, fosters deep collaboration, and enables impactful work at the very heart of the global machine learning community.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [19]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL_BROCHURE,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash-lite
Found 6 relevant links


Hugging Face: The AI Community Building the Future

Hugging Face is the central collaboration platform for the machine learning community, empowering the next generation of engineers, scientists, and end-users to build an open and ethical AI future together. We are at the heart of the AI revolution, fostering a fast-growing community around some of the most used open-source ML libraries and tools.

**What We Offer:**

Hugging Face provides a comprehensive platform for creating, discovering, and collaborating on machine learning. Our key offerings include:

*   **Models:** Explore and utilize over 2 million models, including trending options like Google Gemma.
*   **Datasets:** Access and share over 500,000 datasets to fuel your ML projects.
*   **Spaces:** Host and run over 1 million AI applications, from voice cloning to image generation, with advanced compute options like ZeroGPU.
*   **Buckets:** Benefit from AI-native object storage for your projects.
*   **Documentation & Tools:** Leverage our robust open-source stack and extensive documentation to accelerate your development.

**For Our Customers:**

We serve the entire machine learning community, from individual enthusiasts to large enterprises.

*   **Individuals & Researchers:** Discover, share, and experiment with open-source ML models, datasets, and applications, moving faster with our open-source stack and exploring all modalities of AI.
*   **Organizations (Team & Enterprise Plans):** Scale your AI initiatives with enterprise-grade security, access controls, and dedicated support. Features include Single Sign-On, regional data management, audit logs, granular resource groups, centralized token management, analytics, advanced compute options, private storage, and private dataset viewing.

**Our Culture & Mission:**

At Hugging Face, we are driven by a commitment to foster an open and ethical AI future. We believe in collaboration and empowering individuals and teams to share, explore, discover, and experiment with cutting-edge ML. Our talented science team continuously explores the edge of technology, keeping us at the forefront of AI innovation.

**Careers:**

Join our mission! Hugging Face is constantly growing and seeking passionate individuals to contribute to our vibrant community and groundbreaking work in AI. Visit our careers page for current openings and to learn how you can become a part of building the future of AI.

Hugging Face: Where the world builds AI together.

In [21]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash-lite
Found 9 relevant links


## Hugging Face: The AI Community Building the Future

Hugging Face is the home of machine learning, a leading platform where the global AI community collaborates to create, discover, and accelerate advancements in artificial intelligence. We empower developers, researchers, and enterprises to build the future of AI by providing unparalleled access to powerful tools and resources.

### Our Collaborative Ecosystem

Hugging Face fosters an open and dynamic environment for machine learning innovation through:

*   **Models:** Explore and utilize over 2.7 million cutting-edge machine learning models, ranging from text generation and image processing to any-to-any tasks. Our vast library supports various parameters, frameworks like PyTorch and TensorFlow, and integrations with popular inference providers.
*   **Datasets:** Access nearly 1 million diverse datasets across modalities such as text, image, audio, video, and more. These datasets are crucial for training and evaluating AI models, with options for various formats and sizes.
*   **Spaces:** Discover or host over 1 million interactive AI applications. These "Spaces" allow users to showcase, experiment with, and deploy AI models directly, offering hands-on experiences like high-quality voice cloning (OmniVoice), generating videos from images, or advanced image editing.
*   **Buckets:** Leverage AI-native object storage solutions, optimized for the unique demands of machine learning workflows.

### Empowering Innovation and Collaboration

Hugging Face is more than just a repository; it's a vibrant community platform designed for seamless collaboration. We provide the infrastructure for hosting unlimited public models, datasets, and applications, enabling faster development cycles through our robust open-source stack. Our platform supports the exploration of all modalities, ensuring that innovators have the tools they need to push the boundaries of AI.

### Who We Serve

We serve a diverse audience, including:

*   **The Global AI Community:** Researchers, developers, and students who are passionate about machine learning.
*   **Enterprises:** Companies looking to integrate advanced AI capabilities into their products and services, benefiting from robust models, datasets, and enterprise-grade solutions.
*   **Anyone** looking to move faster and build better with state-of-the-art machine learning.

Join Hugging Face and be part of the community that is shaping the next generation of artificial intelligence.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>